In [2]:
%pip install laspy rasterio open3d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 83.8 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


In [5]:
import numpy as np
import os
from google.colab import drive
import rasterio
import laspy
import open3d as o3d

In [12]:
try:
    # Attempt programmatic mount
    drive.mount('/content/drive')
except Exception as e:
    print("[INFO] Programmatic mount failed. Please click the folder icon on the left sidebar and click 'Mount Drive' manually.")
    print(f"Details: {e}\n")

# Redefine the base path pointing to your mounted Google Drive
# Note: In Colab, 'My Drive' usually mounts as 'MyDrive' or 'My Drive'. We'll use '/content/drive/MyDrive/' as it is the default.
base_data_folder = '/content/drive/MyDrive/Personal_Michael/GeoAI_course_2026/Project_TSG/Data'
images_folder = os.path.join(base_data_folder, 'RGB_Imagery')
lidar_folder = os.path.join(base_data_folder, '700khz_25_FOV')

print("Paths updated to Google Drive mount:")
print(f"Images: {images_folder}")
print(f"LiDAR:  {lidar_folder}")

[INFO] Programmatic mount failed. Please click the folder icon on the left sidebar and click 'Mount Drive' manually.
Details: Error: credential propagation was unsuccessful

Paths updated to Google Drive mount:
Images: /content/drive/MyDrive/Personal_Michael/GeoAI_course_2026/Project_TSG/Data/RGB_Imagery
LiDAR:  /content/drive/MyDrive/Personal_Michael/GeoAI_course_2026/Project_TSG/Data/700khz_25_FOV


In [10]:
# To mount your Drive manually and avoid credential propagation errors:
# 1. Click the Folder (Files) icon on the left sidebar.
# 2. Click the 'Mount Drive' button (folder icon with a Drive logo).
# 3. Once mounted, run the rest of your path-defining scripts.

In [11]:
# Print the content of the folders

def list_folder_contents(folder_name, folder_path):
    print(f"\nChecking {folder_name} at path: {folder_path}")
    if os.path.exists(folder_path):
        try:
            files = os.listdir(folder_path)
            if not files:
                print("   [Folder is empty]")
            else:
                print(f"   Found {len(files)} items:")
                for file in sorted(files):
                    print(f"    - {file}")
        except Exception as e:
            print(f"   Error reading folder: {e}")
    else:
        print("   [Directory does not exist or is not accessible in this environment]")

# List contents of both folders
list_folder_contents("Images Folder", images_folder)
list_folder_contents("LiDAR Folder", lidar_folder)


Checking Images Folder at path: /content/drive/MyDrive/Personal_Michael/GeoAI_course_2026/Project_TSG/Data/RGB_Imagery
   [Directory does not exist or is not accessible in this environment]

Checking LiDAR Folder at path: /content/drive/MyDrive/Personal_Michael/GeoAI_course_2026/Project_TSG/Data/700khz_25_FOV
   [Directory does not exist or is not accessible in this environment]


In [ ]:

def calculate_dsm_lidar_rmse(dsm_tif_path, lidar_las_path):
    """
    Computes the alignment RMSE between a UAV DSM TIFF and a LiDAR LAS subset.
    """
    print("1. Reading UAV DSM raster...")
    with rasterio.open(dsm_tif_path) as src:
        dsm_data = src.read(1)  # Read the first band (elevation)
        transform = src.transform
        nodata = src.nodata

        # Filter out NoData or NaN values to keep only valid surface points
        if nodata is not None:
            valid_mask = (dsm_data != nodata) & ~np.isnan(dsm_data)
        else:
            valid_mask = ~np.isnan(dsm_data)

        rows, cols = np.where(valid_mask)

        # Convert pixel matrix row/col indices into geographic coordinates (X, Y)
        xs, ys = rasterio.transform.xy(transform, rows, cols)
        zs = dsm_data[rows, cols]

        # Stack into an (N, 3) coordinate array
        dsm_points = np.column_stack((xs, ys, zs))
    print(f"   Extracted {len(dsm_points):,} valid pixels from DSM.")

    print("2. Reading LiDAR point cloud subset...")
    las = laspy.read(lidar_las_path)
    # Extract X, Y, Z coordinates from the LAS point cloud
    lidar_points = np.column_stack((las.x, las.y, las.z))
    print(f"   Loaded {len(lidar_points):,} points from LAS subset.")

    print("3. Building Open3D point cloud geometries...")
    pcd_dsm = o3d.geometry.PointCloud()
    pcd_dsm.points = o3d.utility.Vector3dVector(dsm_points)

    pcd_lidar = o3d.geometry.PointCloud()
    pcd_lidar.points = o3d.utility.Vector3dVector(lidar_points)

    print("4. Calculating nearest-neighbor Euclidean distances & RMSE...")
    # Compute distance from each DSM point to the nearest neighbor in the LiDAR cloud
    dists = pcd_dsm.compute_point_cloud_distance(pcd_lidar)
    dists_np = np.asarray(dists)

    # Calculate Root Mean Square Error (RMSE)
    rmse = np.sqrt(np.mean(dists_np ** 2))

    # Optional descriptive statistics
    mean_error = np.mean(dists_np)
    max_error = np.max(dists_np)

    print("\n--- Alignment Results ---")
    print(f"Root Mean Square Error (RMSE): {rmse:.4f} meters")
    print(f"Mean Absolute Distance:        {mean_error:.4f} meters")
    print(f"Maximum Point Deviation:       {max_error:.4f} meters")

    return rmse, dists_np

In [ ]:
    dsm_subset_file = "uav_dsm_subset.tif"
    lidar_subset_file = "lidar_subset.las"

    # Run evaluation
    calculate_dsm_lidar_rmse(dsm_subset_file, lidar_subset_file)